# 🔍 SHAP Explainability Analysis
Understanding WHY the model makes each prediction.

In [ ]:
import joblib
import shap
import pandas as pd
import matplotlib.pyplot as plt
from src.data.loader import load_raw_data
from src.data.preprocessor import clean_data, get_features_and_target
from src.features.engineering import engineer_features
from sklearn.model_selection import train_test_split

df = clean_data(load_raw_data())
df = engineer_features(df)
X, y = get_features_and_target(df)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = joblib.load('models/xgb_v1.pkl')
explainer = joblib.load('models/shap_explainer_v1.pkl')
shap_values = explainer.shap_values(X_test)
print('SHAP values computed for', len(X_test), 'customers')

In [ ]:
# Global feature importance
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test, plot_type='bar', max_display=15, show=False)
plt.title('Global Feature Importance (SHAP)')
plt.tight_layout()
plt.show()

In [ ]:
# SHAP beeswarm plot
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test, max_display=15, show=False)
plt.title('SHAP Value Distribution')
plt.tight_layout()
plt.show()

In [ ]:
# Customer-level explanation — high risk customer
customer_idx = 0
shap_explanation = explainer(X_test.iloc[[customer_idx]])
print(f'Customer: {customer_idx}')
print(f'Churn probability: {model.predict_proba(X_test.iloc[[customer_idx]])[:,1][0]:.1%}')
shap.plots.waterfall(shap_explanation[0], max_display=12, show=False)
plt.tight_layout()
plt.show()

In [ ]:
# Top global features by mean |SHAP|
import numpy as np
mean_shap = pd.Series(np.abs(shap_values).mean(axis=0), index=X_test.columns).sort_values(ascending=False).head(10)
print('Top 10 features by mean |SHAP|:')
print(mean_shap.round(4).to_string())